# 01-02 — Train / Val / Test Split

**Objectif :** appliquer le feature engineering de l’EDA et produire trois splits propres, sans fuite de données, prêts pour la modélisation.

| Paramètre | Valeur | Justification |
|---|---|---|
| Train | 70 % | Suffisant pour TF-IDF + XGBoost |
| Val | 15 % | Tuning Optuna & early stopping |
| Test | 15 % | Évaluation finale — touché une seule fois |
| `stratify` | `target` | Préserve le taux de catastrophe 18,6 % |
| `random_state` | 42 | Reproductibilité |

> **Garde anti-fuite** : `kw_disaster_rate` est calculé sur le **train uniquement**, puis mappé sur val et test.

## 0. Imports globaux

In [1]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from sklearn.model_selection import train_test_split

from settings.params import (
    RANDOM_STATE, TRAIN_SIZE, VAL_SIZE, TEST_SIZE,
    TARGET_COL, FEATURES_ALL, FEATURES_LINEAR,
)
from src.data.make_dataset import load_data
from src.features.build import add_features, add_kw_disaster_rate
from src.utils.logger import logger

pd.set_option('display.max_columns', None)

## 1. Chargement des données brutes

### Description

`load_data` appelle `fetch_openml` et caste les colonnes (`target` → bool, colonnes texte → StringDtype).
Les données brutes ne sont jamais modifiées — toutes les transformations créent de nouvelles colonnes.

In [2]:
data = load_data('disaster-tweets', columns_to_lower=True)
df   = data.copy()
df['label'] = df['target'].map({True: 'Real Disaster', False: 'Not Disaster'})

logger.info(f'Loaded {len(df):,} rows x {df.shape[1]} columns')
df.head(3)

2026-06-03 10:17:06 | INFO | src.data.make_dataset | Loaded 'disaster-tweets' dataset
2026-06-03 10:17:06 | INFO | src.data.make_dataset | Data shape: (11370, 5)
2026-06-03 10:17:06 | INFO | __main__ | Loaded 11,370 rows x 6 columns


,id,keyword,location,text,target,label
0,0,ablaze,<NA>,"Communal violence in Bhainsa, Telangana. ""Ston...",True,Real Disaster
1,1,ablaze,<NA>,Telangana: Section 144 has been imposed in Bha...,True,Real Disaster
2,2,ablaze,New York City,Arsonist sets cars ablaze at dealership https:...,True,Real Disaster


## 2. Feature engineering

`add_features()` reproduit toutes les transformations de l’EDA de manière importable et reproductible.

| Groupe | Features |
|---|---|
| Longueur | `text_length`, `word_count`, `avg_word_length` |
| Binaires | `has_url`, `is_retweet`, `has_numbers`, `has_location` |
| Style | `uppercase_ratio`, `exclamation_count`, `question_count` |
| Tokens (SHAP) | `clean_text`, `tokens`, `token_count`, `unique_tokens`, `ttr` |
| TF-IDF | `preprocessed_text` (stemmé), `preprocessed_len`, `compression_ratio` |

> `kw_disaster_rate` est absent ici — cf. section 4.

In [3]:
df = add_features(df)

logger.info(f'Feature engineering done — {df.shape[1]} colonnes')
df[[
    'text', 'preprocessed_text',
    'text_length', 'word_count', 'preprocessed_len', 'compression_ratio',
    'has_url', 'has_numbers', 'exclamation_count',
]].head(4)

2026-06-03 10:17:16 | INFO | __main__ | Feature engineering done — 24 colonnes


,text,preprocessed_text,text_length,word_count,preprocessed_len,compression_ratio,has_url,has_numbers,exclamation_count
0,"Communal violence in Bhainsa, Telangana. ""Ston...",commun violenc bhainsa telangana stone pelt mu...,124.0,19.0,12.0,0.632,0,0,0
1,Telangana: Section 144 has been imposed in Bha...,telangana section impos bhainsa januari clash ...,130.0,23.0,10.0,0.435,0,1,0
2,Arsonist sets cars ablaze at dealership https:...,arsonist set car ablaz dealership,63.0,7.0,5.0,0.714,1,0,0
3,Arsonist sets cars ablaze at dealership https:...,arsonist set car ablaz dealership,87.0,8.0,5.0,0.625,1,1,0


## 3. Split train / val / test

Stratégie en deux étapes :
1. Dataset complet → **train (70 %)** + temp (30 %)
2. temp → **val (15 %)** + **test (15 %)**

`stratify=target` préserve le taux de catastrophe 18,6 % dans chaque sous-ensemble.

In [4]:
y = df[TARGET_COL].astype(int)

df_train, df_temp = train_test_split(
    df, test_size=(VAL_SIZE + TEST_SIZE),
    stratify=y, random_state=RANDOM_STATE,
)

y_temp = df_temp[TARGET_COL].astype(int)
df_val, df_test = train_test_split(
    df_temp, test_size=0.5,
    stratify=y_temp, random_state=RANDOM_STATE,
)

logger.info(f'train: {len(df_train):,}  val: {len(df_val):,}  test: {len(df_test):,}')

2026-06-03 10:17:16 | INFO | __main__ | train: 7,959  val: 1,705  test: 1,706


## 4. `kw_disaster_rate` sans fuite

Signal dominant (r = 0,44). **Calculé sur train uniquement**, puis mappé sur val et test.
Les mots-clés absents du train reçoivent le taux global d’entraînement.

In [5]:
df_train, df_val, df_test = add_kw_disaster_rate(df_train, df_val, df_test)

assert df_train['kw_disaster_rate'].isna().sum() == 0

for name, split in [('train', df_train), ('val', df_val), ('test', df_test)]:
    print(f"  {name} NaN kw_disaster_rate : {split['kw_disaster_rate'].isna().sum()}")

  train NaN kw_disaster_rate : 0
  val NaN kw_disaster_rate : 0
  test NaN kw_disaster_rate : 0


## 5. Statistiques des splits

In [6]:
rows = []
for name, split in [('train', df_train), ('val', df_val), ('test', df_test)]:
    rows.append({
        'split':         name,
        'n_rows':        len(split),
        'pct_total':     f'{len(split) / len(df) * 100:.1f} %',
        'disaster_rate': f'{split[TARGET_COL].astype(int).mean():.3f}',
    })
pd.DataFrame(rows).set_index('split')

,n_rows,pct_total,disaster_rate
split,,,
train,7959,70.0 %,0.186
val,1705,15.0 %,0.186
test,1706,15.0 %,0.186


### Vérification des features attendues

In [7]:
required = set(FEATURES_ALL) | {'preprocessed_text', 'tokens'}
missing  = required - set(df_train.columns)
if missing:
    raise ValueError(f'Features manquantes dans train : {missing}')

print('Toutes les features requises sont présentes.')
print(f'\nFEATURES_ALL    ({len(FEATURES_ALL)}) : {FEATURES_ALL}')
print(f'\nFEATURES_LINEAR ({len(FEATURES_LINEAR)}) : {FEATURES_LINEAR}')

Toutes les features requises sont présentes.

FEATURES_ALL    (16) : ['kw_disaster_rate', 'has_numbers', 'text_length', 'has_url', 'exclamation_count', 'question_count', 'has_location', 'word_count', 'token_count', 'unique_tokens', 'preprocessed_len', 'ttr', 'compression_ratio', 'avg_word_length', 'uppercase_ratio', 'is_retweet']

FEATURES_LINEAR (7) : ['kw_disaster_rate', 'has_numbers', 'text_length', 'has_url', 'exclamation_count', 'question_count', 'has_location']


## 6. Sauvegarde des splits

Format Parquet (sans perte, préserve les dtypes). Dossier `data/processed/` créé si absent.

In [8]:
out_dir = PROJECT_ROOT / 'data' / 'processed'
out_dir.mkdir(parents=True, exist_ok=True)

for name, split in [('train', df_train), ('val', df_val), ('test', df_test)]:
    path = out_dir / f'{name}.parquet'
    s = split.copy()
    s['tokens'] = s['tokens'].apply(lambda t: ' '.join(t) if isinstance(t, list) else '')
    s.to_parquet(path, index=False)
    logger.info(f'Saved {name} -> {path}  ({len(s):,} lignes)')

print('Splits sauvegardés dans data/processed/')

2026-06-03 10:17:17 | INFO | __main__ | Saved train -> C:\Users\BachirTraore\Desktop\tps\NLP_Disaster_Tweets-\data\processed\train.parquet  (7,959 lignes)
2026-06-03 10:17:17 | INFO | __main__ | Saved val -> C:\Users\BachirTraore\Desktop\tps\NLP_Disaster_Tweets-\data\processed\val.parquet  (1,705 lignes)
2026-06-03 10:17:17 | INFO | __main__ | Saved test -> C:\Users\BachirTraore\Desktop\tps\NLP_Disaster_Tweets-\data\processed\test.parquet  (1,706 lignes)


Splits sauvegardés dans data/processed/


## 7. Récapitulatif

| Fichier | Contenu | Usage |
|---|---|---|
| `data/processed/train.parquet` | 70 % | Entraînement des modèles |
| `data/processed/val.parquet` | 15 % | Tuning Optuna |
| `data/processed/test.parquet` | 15 % | Évaluation finale (touché une seule fois) |

**Prochain notebook :** `disastertweets_02_modeling.ipynb` — baselines (LR, NB, SGD) + Optuna + MLflow.